---
# INFORMS TutORial text example

Section 4.3 illustrative example: knapsack bandit with a feasibility network relaxation.
---


## Modules and Initial Setup

To set up required packages when using Conda, use:

`conda env create -f environment.yml`


In [ ]:
import inspect
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import clear_output, display


def find_project_root(start_path: Path) -> Path:
    """Find the weakly-coupled MDP project root from a notebook directory."""
    for candidate in (start_path, *start_path.parents):
        if (candidate / "wmdp.py").exists() and (candidate / "fnr.py").exists():
            return candidate
    raise RuntimeError("Could not locate the weakly-coupled MDP project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from wmdp import *
from fnr import *
from delayedallocation import *
from lagrangian import *
from simulator import Simulator

print("Project root:", PROJECT_ROOT)


## WMDP Example

This notebook implements the illustrative example in Section 4.3 of *Self-Adapting Networks for Weakly Coupled Dynamic Programs*. The instance is the three-product knapsack bandit from Example 3. In each decision period, action `1` means include a product in the assortment and action `0` means leave it out.

The shared knapsack constraints are

$$2a_1 + 10a_2 + 8a_3 \le 14, \qquad 3a_1 + 18a_2 + 15a_3 \le 20.$$

These constraints allow any single product and the pair of products 1 and 3, but exclude products 1 and 2, products 2 and 3, and all three products together. The example is constructed so the FNR policy chooses product 2 in both decision periods and attains value 49, while the Lagrangian greedy policy can be substantially worse.


## Constructing the WMDP

The next cells translate the Section 4.3 knapsack-bandit instance into the tutorial code: three binary component MDPs, two knapsack linking constraints, and a zero-reward terminal period used only so the simulator executes the two decision periods described in the paper.


### Action Set

Each product has the same binary action set. Action `0` means the product is not included in the assortment; action `1` means it is included.


In [ ]:
# J is the number of product components.
# T includes periods 0 and 1 from Section 4.3 plus a zero-reward terminal period for simulation.
J = 3
T = 3
FIRST_DECISION_PERIODS = [0, 1]
action_sets = [[0, 1] for _ in range(J)]

print("Action sets:")
action_sets


### Component MDPs

All products start in state `s0`. If product `j` is included in period 0, it may transition to state `s1` in period 1; if it is not included, it remains in `s0`. The transition probabilities and rewards match Section 4.3. Product 2 has the largest immediate reward in `s0`, while product 1 has the largest reward after transitioning to `s1`.


In [ ]:
# Rewards for including each product in period 0 state s0.
period0_include_reward = [1.0, 22.0, 0.5]

# Rewards for including each product in period 1 state s1.
period1_high_state_reward = [42.0, 32.0, 11.0]

# Probability of moving from s0 to s1 when the product is included in period 0.
activation_probability = [0.5, 0.5, 1.0]

components = []
for j in range(J):
    state_reward_by_period = [
        [("s0", {0: 0.0, 1: period0_include_reward[j]})],
        [
            ("s0", {0: 0.0, 1: period0_include_reward[j]}),
            ("s1", {0: 0.0, 1: period1_high_state_reward[j]}),
        ],
        [
            ("s0", {0: 0.0, 1: 0.0}),
            ("s1", {0: 0.0, 1: 0.0}),
        ],
    ]

    p = activation_probability[j]
    period0_transition_kernel = {
        ("s0", 0, "s0"): 1.0,
        ("s0", 1, "s0"): 1.0 - p,
        ("s0", 1, "s1"): p,
    }
    period1_terminal_kernel = {
        ("s0", 0, "s0"): 1.0,
        ("s0", 1, "s0"): 1.0,
        ("s1", 0, "s1"): 1.0,
        ("s1", 1, "s1"): 1.0,
    }

    component = build_component(
        component=j,
        actions=action_sets[j],
        state_data_by_period=state_reward_by_period,
        transitions_by_period=[period0_transition_kernel, period1_terminal_kernel],
    )
    components.append(component)

print("Product 2 period-0 state:")
print(components[1].states[0])
print()
print("Product 2 transition kernel from period 0 to period 1:")
print(components[1].P[0])


### Linking Constraints

The next cell adds the two knapsack constraints from Example 3 and Section 4.3. Only action `1` consumes capacity; action `0` has coefficient zero.


In [ ]:
# Coefficients in the two knapsack constraints:
#   2a1 + 10a2 + 8a3 <= 14
#   3a1 + 18a2 + 15a3 <= 20
# Component indices are zero based, so product 1 is component 0.
size_capacity = {
    (0, 1): 2.0,
    (1, 1): 10.0,
    (2, 1): 8.0,
}
weight_capacity = {
    (0, 1): 3.0,
    (1, 1): 18.0,
    (2, 1): 15.0,
}

linking_constraints = build_linking_constraints(
    action_sets=action_sets,
    constraint_coefficients=[size_capacity, weight_capacity],
    rhs_values=[14.0, 20.0],
)

print("Linking constraints:")
print(linking_constraints)


### Assembling the WMDP

The next cell combines the three component MDPs with the shared knapsack constraints. This is the Section 4.3 WMDP instance used by the Lagrangian relaxation and the feasibility network relaxation below.


In [ ]:
# Combine the patient component MDPs and the shared resource constraints into one WMDP.
wmdp = build_wmdp(
    components=components,
    linking_constraints=linking_constraints,
)

print(wmdp)


### The WMDP size

The feasible joint actions are exactly the action vectors permitted by the two knapsack constraints. In this instance they are `(0,0,0)`, each singleton product, and the pair `(1,0,1)`.


In [ ]:
# Count the joint states and joint actions that an exact dynamic program or exact LP would enumerate.
state_counts_by_period = {
    t: len(wmdp.generate_states(t))
    for t in range(wmdp.T)
}

# The feasible joint action space keeps only outreach plans satisfying the two resource budgets.
feasible_actions = [
    tuple(action)
    for action, _ in wmdp.generate_feasible_actions()
]
feasible_action_count = len(feasible_actions)

# An exact recursion or exact LP would consider every feasible action in every joint state.
state_action_counts_by_period = {
    t: state_counts_by_period[t] * feasible_action_count
    for t in range(0, wmdp.T)
}

total_joint_states = sum(state_counts_by_period.values())
total_state_action_pairs = sum(state_action_counts_by_period.values())

print("Exact state-action enumeration using feasible actions:")
for t, count in state_action_counts_by_period.items():
    print(f"  period {t}: {state_counts_by_period[t]} states x {feasible_action_count} actions = {count}")
print(f"Total state-action pairs across {wmdp.T} periods: {total_state_action_pairs}")
print()

## The baseline: Lagrangian relaxation

As a baseline, we solve the expectation-relaxed Lagrangian model. This model enforces the two knapsack constraints in expectation over marginal component flows, so its optimistic bound can be loose for the Section 4.3 instance.


### <b>Construct</b>: Building the Lagrangian Relaxation

The next cell constructs the expectation-relaxed model for the knapsack bandit. It keeps product-level marginal flows and replaces pathwise knapsack feasibility with expected capacity constraints in each period.


In [ ]:
# Construct the expectation-relaxed Lagrangian model.
# The model keeps component-level marginal flows and enforces resources only in expectation.
lagrangian_model = Lagrangian(wmdp)


### <b>Optimize</b>: Solving the Lagrangian Relaxation

The next cell solves the relaxed LP and reports expected capacity use. Section 4.3 reports a Lagrangian bound of about 69.667 for the two decision periods; the zero-reward terminal period does not affect that value.


In [ ]:
# Solve the Lagrangian relaxation and inspect expected resource use by period and constraint.
lagrangian_result = lagrangian_model.optimize()

print("Objective value: ", lagrangian_result.objective_value)

print("\nExpected resource use:")
for (period, constraint), value in lagrangian_result.expected_resource_use.items():
    budget = wmdp.linking_constraints.b[constraint]
    print(f"  period {period}, constraint {constraint}: {value:.3f} / {budget:.3f}")


### Simulating the Lagrangian Policy

The tutorial policy samples from the Lagrangian marginals and repairs infeasible sampled actions online. This is a feasible executable policy derived from the relaxation, not the exact greedy policy calculation reported in the paper.


In [ ]:
# Simulate the repaired Lagrangian policy and plot total realized rewards over the two decision periods.
num_simulations = 100
lagrangian_simulator = Simulator(wmdp, lagrangian_result.policy)
lagrangian_total_rewards = []
lagrangian_action_paths = []

fig, ax = plt.subplots(figsize=(8, 4))

for simulation_index in range(num_simulations):
    simulation_result = lagrangian_simulator.simulate()
    lagrangian_total_rewards.append(simulation_result["total_reward"])
    lagrangian_action_paths.append(tuple(tuple(action) for action in simulation_result["actions"]))

    ax.clear()
    ax.hist(
        lagrangian_total_rewards,
        bins=min(30, max(1, len(set(lagrangian_total_rewards)))),
        color="steelblue",
        edgecolor="white",
    )
    sample_mean = sum(lagrangian_total_rewards) / len(lagrangian_total_rewards)
    ax.axvline(
        sample_mean,
        color="darkorange",
        linewidth=2,
        label=f"mean: {sample_mean:.2f}",
    )
    ax.axvline(
        lagrangian_result.objective_value,
        color="seagreen",
        linewidth=2,
        linestyle="--",
        label=f"relaxation: {lagrangian_result.objective_value:.2f}",
    )
    ax.set_title(f"Lagrangian policy simulation ({simulation_index + 1}/{num_simulations})")
    ax.set_xlabel("Total reward")
    ax.set_ylabel("Frequency")
    ax.legend(loc="upper right")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

print(f"Mean total reward over {num_simulations} simulations: {sum(lagrangian_total_rewards) / len(lagrangian_total_rewards):.3f}")
print("Most common action paths:")
for path, count in sorted(
    {path: lagrangian_action_paths.count(path) for path in set(lagrangian_action_paths)}.items(),
    key=lambda item: item[1],
    reverse=True,
)[:5]:
    print(f"  {path}: {count}")


## Self-Adapting Network Relaxations

We now build and solve the feasible network relaxation (FNR) for the same knapsack bandit. The network represents exactly the feasible product-assortment choices under the two knapsack constraints.


### <b>Construct</b>: Building the Feasibility Network Relaxation

The next cell constructs the reduced layered network from the two knapsack constraints. Each root-to-terminal path corresponds to one feasible action vector.


In [ ]:
# Build the FNR network from the two knapsack constraints.
# Network paths encode feasible product-assortment decisions.
network = construct_fnr_network(wmdp.linking_constraints)
print("Network size:")
print(network.get_size())


### Visualizing the Network

The next cell draws the feasibility network. Layers correspond to products, and arcs correspond to excluding or including a product while tracking remaining knapsack capacity.


In [ ]:
# Draw the layered feasibility network for the Section 4.3 knapsack instance.
draw_fnr_network(network, figsize=(10, 3))


<b>How many actions does this represent?</b>

The next code enumerates all feasible actions, each represented as a path of the network.

In [ ]:
feasible_actions = [
    tuple(action)
    for action, _ in wmdp.generate_feasible_actions()
]

print(f"Feasible actions (total: {len(feasible_actions)}):")
for action in feasible_actions:
    print("\tAction " + str(action))

<b>Linear programming construction</b>

The next cell constructs the FNR LP over the same component flows, linked to network-arc flows that encode feasible action vectors.


In [ ]:
# Construct the FNR linear-programming model for the Section 4.3 knapsack bandit.
FNR_model = FNR(wmdp, network)


## <b>Optimize</b>: Solving the FNR Model

The next cell solves the FNR linear program. Section 4.3 reports an FNR bound of 49, obtained by choosing product 2 in both periods and accounting for its period-1 transition.


In [ ]:
# Solve the FNR model and report the expected total reward and positive marginal action flows.
fnr_result = FNR_model.optimize()

print("Objective value: ", fnr_result.objective_value)
print("Positive marginals:")
for key, value in sorted(fnr_result.marginal_flows.items()):
    if value > 1e-8:
        print(f"  {key}: {value:.3f}")


### Simulating the FNR Policy

The FNR policy follows positive network flow when available. For this instance, Section 4.3 predicts action `(0,1,0)` in both decision periods for every state reachable from the initial state.


In [ ]:
# Simulate the FNR policy and plot total realized rewards over the two decision periods.
num_simulations = 100
fnr_simulator = Simulator(wmdp, fnr_result.policy)
fnr_total_rewards = []
fnr_action_paths = []

fig, ax = plt.subplots(figsize=(8, 4))

for simulation_index in range(num_simulations):
    simulation_result = fnr_simulator.simulate()
    fnr_total_rewards.append(simulation_result["total_reward"])
    fnr_action_paths.append(tuple(tuple(action) for action in simulation_result["actions"]))

    ax.clear()
    ax.hist(
        fnr_total_rewards,
        bins=min(30, max(1, len(set(fnr_total_rewards)))),
        color="steelblue",
        edgecolor="white",
    )
    sample_mean = sum(fnr_total_rewards) / len(fnr_total_rewards)
    ax.axvline(
        sample_mean,
        color="darkorange",
        linewidth=2,
        label=f"mean: {sample_mean:.2f}",
    )
    ax.axvline(
        fnr_result.objective_value,
        color="seagreen",
        linewidth=2,
        linestyle="--",
        label=f"model: {fnr_result.objective_value:.2f}",
    )
    ax.set_title(f"FNR policy simulation ({simulation_index + 1}/{num_simulations})")
    ax.set_xlabel("Total reward")
    ax.set_ylabel("Frequency")
    ax.legend(loc="upper right")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

clear_output(wait=True)
display(fig)
print(f"Mean total reward over {num_simulations} simulations: {sum(fnr_total_rewards) / len(fnr_total_rewards):.3f}")
print("Action paths observed:")
for path, count in sorted(
    {path: fnr_action_paths.count(path) for path in set(fnr_action_paths)}.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"  {path}: {count}")


## Delayed Allocation

We now solve the delayed-allocation model for the same Section 4.3 knapsack bandit. The model starts with a deliberately small action set containing only the all-zero action `(0, 0, 0)` in every period, then performs exactly three refinement iterations using the exact linking-IP separation oracle.


### <b>Construct</b>: Initial Restricted Action Set

The next cell initializes delayed allocation with one feasible joint action in each period: exclude every product. Refinement can then add feasible actions with positive reduced cost.


In [ ]:
# Start delayed allocation with one available joint action in each period: exclude every product.
initial_actions = {
    period: [tuple(0 for _ in range(J))]
    for period in range(wmdp.T)
}

print("Initial delayed-allocation actions:")
for period, actions in initial_actions.items():
    print(f"  period {period}: {actions}")


In [ ]:
# Build and solve the restricted delayed-allocation model.
DA_model = DelayedAllocationModel(wmdp, initial_actions=initial_actions)
delayed_allocation_result = DA_model.optimize()

print("Initial restricted objective value:", delayed_allocation_result.objective_value)
print("Positive pi actions:")
for period, actions in delayed_allocation_result.positive_pi_actions.items():
    print(f"  period {period}: {actions}")


### <b>Refine</b>: Three Delayed-Allocation Iterations

Each refinement solves a pricing problem for each period and adds at most one improving feasible joint action per period. The loop below intentionally stops after three iterations, even if more refinement were possible.


In [ ]:
# Run exactly three delayed-allocation refinement iterations.
exact_separation = LinkingIPSeparation()

def snapshot_delayed_allocation(iteration, result):
    action_sets = {
        period: list(actions)
        for period, actions in DA_model.action_set.items()
    }
    positive_actions = {
        period: [action for action, value in actions if value > DA_model.tolerance]
        for period, actions in result.positive_pi_actions.items()
    }
    return {
        "iteration": iteration,
        "objective": result.objective_value,
        "num_new_actions": result.num_new_actions,
        "added_actions": result.added_actions,
        "action_sets": action_sets,
        "positive_actions": positive_actions,
        "policy": result.policy,
        "action_set_size": {period: len(actions) for period, actions in action_sets.items()},
    }

delayed_allocation_history = [snapshot_delayed_allocation(0, delayed_allocation_result)]

for iteration in range(1, 4):
    delayed_allocation_result = DA_model.refine(
        duals=delayed_allocation_result.duals,
        separation_method=exact_separation,
        verbose=True,
    )
    delayed_allocation_history.append(snapshot_delayed_allocation(iteration, delayed_allocation_result))

print("Delayed-allocation refinement history:")
for row in delayed_allocation_history:
    print(
        f"  iteration {row['iteration']}: "
        f"objective={row['objective']:.3f}, "
        f"new_actions={row['num_new_actions']}, "
        f"action_set_size={row['action_set_size']}"
    )


In [ ]:
# Inspect the actions available after three refinement iterations.
print("Delayed-allocation action sets after three iterations:")
for period, actions in DA_model.action_set.items():
    print(f"  period {period}:")
    for action in actions:
        marker = "*" if any(
            positive_action == action
            for positive_action, _ in delayed_allocation_result.positive_pi_actions[period]
        ) else " "
        print(f"    {marker} {action}")

print("\nPositive pi actions in the final restricted solution:")
for period, actions in delayed_allocation_result.positive_pi_actions.items():
    print(f"  period {period}: {actions}")


### Extreme Points of the Linear Relaxation

Before plotting the delayed-allocation action hulls, we can inspect the LP relaxation of the two knapsack constraints. The relaxation replaces the binary requirements with bounds $0 \le a_j \le 1$ and keeps the same linking constraints. The next cell enumerates its extreme points by intersecting triples of active inequalities.


In [ ]:
# Enumerate extreme points of the LP relaxation:
#   0 <= a_j <= 1
#   2a1 + 10a2 + 8a3 <= 14
#   3a1 + 18a2 + 15a3 <= 20
from itertools import combinations

import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection



def vector_subtract(left, right):
    return tuple(left[index] - right[index] for index in range(3))


def cross_product(left, right):
    return (
        left[1] * right[2] - left[2] * right[1],
        left[2] * right[0] - left[0] * right[2],
        left[0] * right[1] - left[1] * right[0],
    )


def dot_product(left, right):
    return sum(left[index] * right[index] for index in range(3))


def convex_hull_faces(points, tolerance=1e-9):
    """Return triangular hull faces for a small set of 3D points."""
    if len(points) < 3:
        return []

    faces = []
    seen = set()
    for face_indices in combinations(range(len(points)), 3):
        p0, p1, p2 = [points[index] for index in face_indices]
        normal = cross_product(vector_subtract(p1, p0), vector_subtract(p2, p0))
        if dot_product(normal, normal) <= tolerance:
            continue

        signed_distances = [
            dot_product(normal, vector_subtract(point, p0))
            for point in points
        ]
        if all(distance >= -tolerance for distance in signed_distances) or all(
            distance <= tolerance for distance in signed_distances
        ):
            key = tuple(sorted(face_indices))
            if key not in seen:
                seen.add(key)
                faces.append([p0, p1, p2])
    return faces

relaxation_inequalities = [
    (np.array([1.0, 0.0, 0.0]), 1.0, "a1 <= 1"),
    (np.array([0.0, 1.0, 0.0]), 1.0, "a2 <= 1"),
    (np.array([0.0, 0.0, 1.0]), 1.0, "a3 <= 1"),
    (np.array([-1.0, 0.0, 0.0]), 0.0, "a1 >= 0"),
    (np.array([0.0, -1.0, 0.0]), 0.0, "a2 >= 0"),
    (np.array([0.0, 0.0, -1.0]), 0.0, "a3 >= 0"),
    (np.array([2.0, 10.0, 8.0]), 14.0, "2a1 + 10a2 + 8a3 <= 14"),
    (np.array([3.0, 18.0, 15.0]), 20.0, "3a1 + 18a2 + 15a3 <= 20"),
]

def is_relaxation_feasible(point, tolerance=1e-9):
    return all(
        normal @ point <= rhs + tolerance
        for normal, rhs, _ in relaxation_inequalities
    )

linear_relaxation_extreme_points = []
active_constraint_sets = []
seen = set()

for active_indices in combinations(range(len(relaxation_inequalities)), 3):
    matrix = np.vstack([relaxation_inequalities[index][0] for index in active_indices])
    rhs = np.array([relaxation_inequalities[index][1] for index in active_indices])
    if np.linalg.matrix_rank(matrix) < 3:
        continue

    point = np.linalg.solve(matrix, rhs)
    if not is_relaxation_feasible(point):
        continue

    rounded_key = tuple(np.round(point, 10))
    if rounded_key in seen:
        continue

    seen.add(rounded_key)
    linear_relaxation_extreme_points.append(tuple(float(value) for value in point))
    active_constraint_sets.append(
        tuple(relaxation_inequalities[index][2] for index in active_indices)
    )

linear_relaxation_extreme_points = sorted(linear_relaxation_extreme_points)

print("Extreme points of the LP relaxation:")
for point in linear_relaxation_extreme_points:
    integral_marker = "integral" if all(abs(value - round(value)) <= 1e-9 for value in point) else "fractional"
    print(f"  {tuple(round(value, 4) for value in point)}  ({integral_marker})")

fig = plt.figure(figsize=(7, 6))
axis = fig.add_subplot(111, projection="3d")

integral_points = [
    point for point in linear_relaxation_extreme_points
    if all(abs(value - round(value)) <= 1e-9 for value in point)
]
fractional_points = [
    point for point in linear_relaxation_extreme_points
    if point not in integral_points
]

if integral_points:
    axis.scatter(
        [point[0] for point in integral_points],
        [point[1] for point in integral_points],
        [point[2] for point in integral_points],
        s=80,
        color="seagreen",
        edgecolors="black",
        linewidths=0.8,
        label="integral extreme point",
    )
if fractional_points:
    axis.scatter(
        [point[0] for point in fractional_points],
        [point[1] for point in fractional_points],
        [point[2] for point in fractional_points],
        s=90,
        color="crimson",
        marker="^",
        edgecolors="black",
        linewidths=0.8,
        label="fractional extreme point",
    )

faces = convex_hull_faces(linear_relaxation_extreme_points)
if faces:
    hull = Poly3DCollection(
        faces,
        facecolor="lightsteelblue",
        edgecolor="none",
        linewidth=0.0,
        alpha=0.18,
    )
    axis.add_collection3d(hull)

for point in linear_relaxation_extreme_points:
    label = tuple(round(value, 2) for value in point)
    axis.text(point[0], point[1], point[2] + 0.04, str(label), fontsize=8)

axis.set_title("LP relaxation extreme points")
axis.set_xlabel("component 1 action")
axis.set_ylabel("component 2 action")
axis.set_zlabel("component 3 action")
axis.set_xlim(-0.1, 1.1)
axis.set_ylim(-0.1, 1.1)
axis.set_zlim(-0.1, 1.1)
axis.set_xticks([0, 1])
axis.set_yticks([0, 1])
axis.set_zticks([0, 1])
axis.view_init(elev=24, azim=38)
axis.legend(loc="upper left")
plt.tight_layout()
plt.show()


### Convex Hulls of Generated Actions

The next cell plots the cumulative action set available after each delayed-allocation iteration. Each 3D axis is one product component. Orange points are actions with positive `pi` in at least one period at that iteration; gray hollow points are generated actions with zero `pi` in every period. The translucent surface is the convex hull of the generated action points whenever enough non-collinear points are available.


In [ ]:
# Plot the convex hull of generated actions after each delayed-allocation iteration.
from itertools import combinations

from mpl_toolkits.mplot3d.art3d import Poly3DCollection


def unique_actions_from_periods(action_sets_by_period):
    return sorted({tuple(action) for actions in action_sets_by_period.values() for action in actions})


def positive_actions_from_periods(positive_actions_by_period):
    return {tuple(action) for actions in positive_actions_by_period.values() for action in actions}


def vector_subtract(left, right):
    return tuple(left[index] - right[index] for index in range(3))


def cross_product(left, right):
    return (
        left[1] * right[2] - left[2] * right[1],
        left[2] * right[0] - left[0] * right[2],
        left[0] * right[1] - left[1] * right[0],
    )


def dot_product(left, right):
    return sum(left[index] * right[index] for index in range(3))


def convex_hull_faces(points, tolerance=1e-9):
    """Return triangular hull faces for a small set of 3D points."""
    if len(points) < 3:
        return []

    faces = []
    seen = set()
    for face_indices in combinations(range(len(points)), 3):
        p0, p1, p2 = [points[index] for index in face_indices]
        normal = cross_product(vector_subtract(p1, p0), vector_subtract(p2, p0))
        if dot_product(normal, normal) <= tolerance:
            continue

        signed_distances = [
            dot_product(normal, vector_subtract(point, p0))
            for point in points
        ]
        if all(distance >= -tolerance for distance in signed_distances) or all(
            distance <= tolerance for distance in signed_distances
        ):
            key = tuple(sorted(face_indices))
            if key not in seen:
                seen.add(key)
                faces.append([p0, p1, p2])
    return faces


def draw_hull_edges(axis, points):
    for left, right in combinations(points, 2):
        axis.plot(
            [left[0], right[0]],
            [left[1], right[1]],
            [left[2], right[2]],
            color="0.55",
            linewidth=0.8,
            alpha=0.7,
        )


fig = plt.figure(figsize=(12, 10))

for plot_index, row in enumerate(delayed_allocation_history, start=1):
    axis = fig.add_subplot(2, 2, plot_index, projection="3d")
    actions = unique_actions_from_periods(row["action_sets"])
    positive_actions = positive_actions_from_periods(row["positive_actions"])
    non_positive_actions = [action for action in actions if action not in positive_actions]

    faces = convex_hull_faces(actions)
    if faces:
        hull = Poly3DCollection(
            faces,
            facecolor="cornflowerblue",
            edgecolor="none",
            linewidth=0.0,
            alpha=0.18,
        )
        axis.add_collection3d(hull)
    if non_positive_actions:
        axis.scatter(
            [action[0] for action in non_positive_actions],
            [action[1] for action in non_positive_actions],
            [action[2] for action in non_positive_actions],
            s=70,
            facecolors="white",
            edgecolors="0.35",
            linewidths=1.4,
            label="non-positive pi",
        )
    if positive_actions:
        positive_list = sorted(positive_actions)
        axis.scatter(
            [action[0] for action in positive_list],
            [action[1] for action in positive_list],
            [action[2] for action in positive_list],
            s=90,
            color="darkorange",
            edgecolors="black",
            linewidths=0.8,
            label="positive pi",
        )

    for action in actions:
        axis.text(action[0], action[1], action[2] + 0.04, str(action), fontsize=8)

    axis.set_title(
        f"Iteration {row['iteration']} | objective {row['objective']:.1f}"
    )
    axis.set_xlabel("component 1 action")
    axis.set_ylabel("component 2 action")
    axis.set_zlabel("component 3 action")
    axis.set_xlim(-0.1, 1.1)
    axis.set_ylim(-0.1, 1.1)
    axis.set_zlim(-0.1, 1.1)
    axis.set_xticks([0, 1])
    axis.set_yticks([0, 1])
    axis.set_zticks([0, 1])
    axis.view_init(elev=20, azim=90)
    axis.legend(loc="upper left")

plt.tight_layout()
plt.show()

print("Generated actions by delayed-allocation iteration:")
for row in delayed_allocation_history:
    actions = unique_actions_from_periods(row["action_sets"])
    positive_actions = positive_actions_from_periods(row["positive_actions"])
    non_positive_actions = [action for action in actions if action not in positive_actions]
    print(f"  iteration {row['iteration']}:")
    print(f"    positive pi:     {sorted(positive_actions)}")
    print(f"    non-positive pi: {non_positive_actions}")


### Simulating the Delayed-Allocation Policy

The policy induced by the delayed-allocation solution samples from the positive restricted action vertices in each period and state. After three refinement iterations, compare its realized value with the Lagrangian and FNR policies.


In [ ]:
# Simulate the delayed-allocation policy after three refinement iterations.
num_simulations = 100
delayed_allocation_simulator = Simulator(wmdp, delayed_allocation_result.policy)
delayed_allocation_total_rewards = []
delayed_allocation_action_paths = []

fig, ax = plt.subplots(figsize=(8, 4))

for simulation_index in range(num_simulations):
    simulation_result = delayed_allocation_simulator.simulate()
    delayed_allocation_total_rewards.append(simulation_result["total_reward"])
    delayed_allocation_action_paths.append(tuple(tuple(action) for action in simulation_result["actions"]))

    ax.clear()
    ax.hist(
        delayed_allocation_total_rewards,
        bins=min(30, max(1, len(set(delayed_allocation_total_rewards)))),
        color="steelblue",
        edgecolor="white",
    )
    sample_mean = sum(delayed_allocation_total_rewards) / len(delayed_allocation_total_rewards)
    ax.axvline(
        sample_mean,
        color="darkorange",
        linewidth=2,
        label=f"mean: {sample_mean:.2f}",
    )
    ax.axvline(
        delayed_allocation_result.objective_value,
        color="seagreen",
        linewidth=2,
        linestyle="--",
        label=f"model: {delayed_allocation_result.objective_value:.2f}",
    )
    ax.set_title(f"Delayed-allocation policy simulation ({simulation_index + 1}/{num_simulations})")
    ax.set_xlabel("Total reward")
    ax.set_ylabel("Frequency")
    ax.legend(loc="upper right")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

clear_output(wait=True)
display(fig)
print(f"Mean total reward over {num_simulations} simulations: {sum(delayed_allocation_total_rewards) / len(delayed_allocation_total_rewards):.3f}")
print("Action paths observed:")
for path, count in sorted(
    {path: delayed_allocation_action_paths.count(path) for path in set(delayed_allocation_action_paths)}.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"  {path}: {count}")


## Value of Self-Adaptation Relative to the Baseline

The Section 4.3 FNR bound and policy value are 49. The Lagrangian relaxation has a looser optimistic bound, while feasible policies derived from Lagrangian repair and delayed allocation must be evaluated separately. The comparison below reports model objectives and simulated feasible policy means from this notebook.


In [ ]:
lagrangian_mean = sum(lagrangian_total_rewards) / len(lagrangian_total_rewards)
fnr_mean = sum(fnr_total_rewards) / len(fnr_total_rewards)
delayed_allocation_mean = sum(delayed_allocation_total_rewards) / len(delayed_allocation_total_rewards)

print("Reference values from Section 4.3:")
print("  FNR bound and policy value: 49.000")
print("  Lagrangian bound:           69.667")
print("  Lagrangian greedy policy:   39.000")
print()
print("Model objective values in this notebook:")
print(f"  Lagrangian relaxation: {lagrangian_result.objective_value:.3f}")
print(f"  FNR model:             {fnr_result.objective_value:.3f}")
print(f"  Delayed allocation:    {delayed_allocation_result.objective_value:.3f}")
print()
print("Simulated feasible policy values:")
print(f"  Repaired Lagrangian policy mean: {lagrangian_mean:.3f}")
print(f"  FNR policy mean:                 {fnr_mean:.3f}")
print(f"  Delayed-allocation policy mean:  {delayed_allocation_mean:.3f}")
print(f"  DA improvement over Lagrangian:  {delayed_allocation_mean - lagrangian_mean:.3f}")
print(f"  DA gap versus FNR:               {fnr_mean - delayed_allocation_mean:.3f}")


## Delayed-Allocation Progression Plots

The final plots summarize how delayed allocation improves as actions are generated. The first plot tracks the delayed-allocation model objective by refinement iteration, with the Lagrangian and FNR objectives as reference lines. The second plot evaluates the executable delayed-allocation policy at each iteration and compares it with the repaired Lagrangian policy and the FNR policy.


In [ ]:
# Plot delayed-allocation bound and policy-value progression.
from collections import defaultdict

try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
    print("seaborn is not installed; using a matplotlib fallback for these line plots.")


def evaluate_policy_exactly(wmdp, policy):
    """Compute the exact expected total reward of a deterministic policy."""
    state_distribution = {tuple(wmdp.generate_states(0)[0]): 1.0}
    expected_reward = 0.0

    for period in range(wmdp.T - 1):
        next_distribution = defaultdict(float)
        for state_tuple, state_probability in state_distribution.items():
            state = list(state_tuple)
            action = tuple(policy.get_action(period=period, state=state))
            expected_reward += state_probability * sum(
                component_state.reward[action[component]]
                for component, component_state in enumerate(state)
            )

            component_next_options = []
            for component, component_state in enumerate(state):
                component_model = wmdp.state_space.S[component]
                transition_kernel = component_model.P[period]
                options = []
                for next_state in component_model.states[period + 1]:
                    probability = transition_kernel.get(
                        (component_state.label, action[component], next_state.label),
                        0.0,
                    )
                    if probability > 0.0:
                        options.append((next_state, probability))
                component_next_options.append(options)

            next_states = [([], 1.0)]
            for options in component_next_options:
                expanded = []
                for prefix, prefix_probability in next_states:
                    for next_state, transition_probability in options:
                        expanded.append(
                            (prefix + [next_state], prefix_probability * transition_probability)
                        )
                next_states = expanded

            for next_state, transition_probability in next_states:
                next_distribution[tuple(next_state)] += state_probability * transition_probability

        state_distribution = dict(next_distribution)

    return expected_reward


progression_rows = []
for row in delayed_allocation_history:
    progression_rows.append(
        {
            "iteration": row["iteration"],
            "model_bound": row["objective"],
            "policy_value": evaluate_policy_exactly(wmdp, row["policy"]),
        }
    )

iterations = [row["iteration"] for row in progression_rows]
da_bounds = [row["model_bound"] for row in progression_rows]
da_policy_values = [row["policy_value"] for row in progression_rows]
method_colors = {
    "Delayed allocation": "steelblue",
    "LAG": "crimson",
    "FNR": "seagreen",
}
fnr_policy_value = evaluate_policy_exactly(wmdp, fnr_result.policy)

shared_y_values = (
    da_bounds
    + da_policy_values
    + [
        lagrangian_result.objective_value,
        fnr_result.objective_value,
        lagrangian_mean,
        fnr_policy_value,
    ]
)
y_min = min(shared_y_values)
y_max = max(shared_y_values)
y_padding = max(1.0, 0.06 * (y_max - y_min))
shared_ylim = (y_min - y_padding, y_max + y_padding)

if sns is not None:
    # sns.set_theme(style="whitegrid")
    sns.set_theme(style="darkgrid")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

if sns is not None:
    sns.lineplot(
        x=iterations,
        y=da_bounds,
        marker="o",
        markersize=8,
        linewidth=3.4,
        color=method_colors["Delayed allocation"],
        ax=axes[0],
        label="Delayed allocation",
    )
else:
    axes[0].plot(
        iterations,
        da_bounds,
        marker="o",
        markersize=8,
        linewidth=3.4,
        color=method_colors["Delayed allocation"],
        label="Delayed allocation",
    )
axes[0].axhline(
    lagrangian_result.objective_value,
    color=method_colors["LAG"],
    linestyle="--",
    linewidth=2.6,
    label=f"LAG bound ({lagrangian_result.objective_value:.1f})",
)
axes[0].axhline(
    fnr_result.objective_value,
    color=method_colors["FNR"],
    linestyle=":",
    linewidth=3.0,
    label=f"FNR bound ({fnr_result.objective_value:.1f})",
)
axes[0].set_title("Bound progression")
axes[0].set_xlabel("Delayed-allocation refinement iteration")
axes[0].set_ylabel("Model objective")
axes[0].set_xticks(iterations)
axes[0].set_ylim(shared_ylim)
axes[0].legend(loc="lower right")

if sns is not None:
    sns.lineplot(
        x=iterations,
        y=da_policy_values,
        marker="o",
        markersize=8,
        linewidth=3.4,
        color=method_colors["Delayed allocation"],
        ax=axes[1],
        label="Delayed-allocation policy",
    )
else:
    axes[1].plot(
        iterations,
        da_policy_values,
        marker="o",
        markersize=8,
        linewidth=3.4,
        color=method_colors["Delayed allocation"],
        label="Delayed-allocation policy",
    )
axes[1].axhline(
    lagrangian_mean,
    color=method_colors["LAG"],
    linestyle="--",
    linewidth=2.6,
    label=f"Repaired LAG policy ({lagrangian_mean:.1f})",
)
axes[1].axhline(
    fnr_policy_value,
    color=method_colors["FNR"],
    linestyle=":",
    linewidth=3.0,
    label=f"FNR policy ({fnr_policy_value:.1f})",
)
axes[1].set_title("Policy lower-bound progression")
axes[1].set_xlabel("Delayed-allocation refinement iteration")
axes[1].set_ylabel("Expected policy value")
axes[1].set_xticks(iterations)
axes[1].set_ylim(shared_ylim)
axes[1].legend(loc="lower right")



plt.show()

progression_pdf_path = Path("delayed_allocation_progression.pdf")
fig.savefig(progression_pdf_path, format="pdf", bbox_inches="tight")
print(f"Saved progression plots to {progression_pdf_path.resolve()}")

print("Delayed-allocation progression values:")
for row in progression_rows:
    print(
        f"  iteration {row['iteration']}: "
        f"model_bound={row['model_bound']:.3f}, "
        f"policy_value={row['policy_value']:.3f}"
    )


## Final Model Sizes

The next cell compares the final optimization-model sizes for the Lagrangian relaxation, the FNR model, and the converged delayed-allocation model. The reported size is the number of variables plus the number of constraints in the underlying Gurobi model.


In [ ]:
def current_model_size(model_owner):
    """Return variable, constraint, and total size for a Gurobi-backed model."""
    model = getattr(model_owner, "model", model_owner)
    model.update()
    variables = model.NumVars
    constraints = model.NumConstrs
    return {
        "variables": variables,
        "constraints": constraints,
        "variables_plus_constraints": variables + constraints,
        "nonzeros": model.NumNZs,
    }


model_size_summary = {
    "LAG": current_model_size(lagrangian_model),
    "FNR": current_model_size(FNR_model),
    "Converged DA": current_model_size(DA_model),
}

print("Final model sizes:")
print(f"{'method':<14} {'variables':>10} {'constraints':>12} {'vars+constrs':>14} {'nonzeros':>10}")
for method, size in model_size_summary.items():
    print(
        f"{method:<14} "
        f"{size['variables']:>10} "
        f"{size['constraints']:>12} "
        f"{size['variables_plus_constraints']:>14} "
        f"{size['nonzeros']:>10}"
    )
